# Week 9 — Trust + Evaluation
### *Hallucinations, guardrails, trust UX, and proving your fixes worked.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week9_trust_evaluation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Learning objectives
By the end of these two class sessions, you can:
- Distinguish hallucination types: **unsupported**, **contradicted**, **wrong citation**.
- Explain why “trust” is part of system design (not just model accuracy).
- Implement simple metrics: **attack success rate**, **hallucination rate**, **refusal precision/recall**.
- Explain **Goodhart’s Law** with an example from guardrails.
- Describe how A/B testing differs from offline evaluation.


In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git || true
import sys, platform
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/content/main")
from course_utils import get_text_embedding

# Optional: DSPy
try:
    import dspy
except Exception:
    dspy = None

print(f"✅ Ready! Python {platform.python_version()} | dspy={'yes' if dspy else 'no'}")


---

# Tue 9 — Hallucinations + trust-aware UX

## **What did memory change?**
Memory makes agents more helpful — and also makes mistakes “stick.”

Now we ask: **How should the system communicate uncertainty and evidence?**

---

## **Hallucinations as confident errors**
Three useful categories:

1) **Unsupported**: not in sources
2) **Contradicted**: sources say the opposite
3) **Wrong citation**: cites a source that doesn’t support the claim

A key engineering idea:
> If you can’t *verify* a claim from evidence, your UI should not pretend it’s trustworthy.

---

## **Trust-aware UX patterns**
- Show **sources** (citations)
- Show **tool trace** (“I used RAG because…”, “I computed…”)
- Provide an explicit **abstain** state: “I don’t know based on available evidence.”
- Use warnings for risky actions (“This will email someone”)

### Reflection
> What do you personally want to see before trusting an answer: citations, trace, confidence…?


---

# Thu 9 — Evaluation & metrics

## **Why metrics matter**
After Week 7 (attacks) and Week 8 (memory), we can make changes…
…but did we actually improve?

Metrics let us:
- compare baseline vs improved
- catch regressions
- see tradeoffs (usefulness vs refusal)

---

## A small set of metrics that fit our course systems

### 1) Attack success rate
$$
\frac{\#\text{successful attacks}}{\#\text{total attacks}}
$$

### 2) Refusal precision / recall
Treat “refuse” as a classifier:
- **precision**: of the things you refused, how many *should* be refused?
- **recall**: of the things that *should* be refused, how many did you refuse?

### 3) Hallucination rate (simple proxy)
One proxy:
- fraction of answers that contain **no citation** when citations are required

This is imperfect, but it’s testable.

---

## Goodhart’s Law (the classic trap)
If you optimize “safety” as *refusal rate*, the best score is: **refuse everything**.
That’s safe… and useless.

So we often need multiple metrics:
- usefulness (task success)
- safety (attack success rate)
- trust (citation coverage, abstain correctness)


In [ ]:
# @title Visual: Goodhart tradeoff curve (toy)
x = np.linspace(0, 1, 50)  # "strictness" of guardrails
usefulness = 1 - 0.9*x
safety = x**0.8

plt.figure(figsize=(6,3))
plt.plot(x, usefulness, label="usefulness")
plt.plot(x, safety, label="safety")
plt.xlabel("guardrail strictness (toy)")
plt.ylabel("score")
plt.title("Goodhart intuition: one metric isn't enough")
plt.legend()
plt.tight_layout()
plt.show()


---

## A/B testing basics (tiny)
Offline evaluation:
- fixed dataset
- repeatable
- quick iteration

Online A/B testing:
- real users
- compares two variants in production
- needs careful logging + ethics + safety checks

We’ll mostly do offline evaluation in this course, but you should know the idea exists.

---

## (Optional) DSPy: LLM-as-judge (carefully)
DSPy can help structure a “judge” that checks:
- does the answer cite a source?
- does the cited source contain the claim?

But: automated judges can be wrong, so we use them as **assistants**, not final truth.

---

<details>
<summary><strong>Instructor Notes</strong></summary>

### Tue pacing
- 0–10: Lab 8 review (when memory helped/hurt)
- 10–25: hallucination taxonomy + examples
- 25–40: trust UX patterns (citations, trace, abstain)
- 40–50: connect to Lab 9 eval harness

### Thu pacing
- 0–20: key metrics (attack SR, refusal P/R, hallucination proxy)
- 20–30: Goodhart story + tradeoff curve
- 30–40: human vs automated eval (brief)
- 40–50: A/B testing basics + Lab 9 kickoff

</details>
